# PROCESSING LABELS TO TIMESERIES

Based on the information given in the raw .txt files on the anomalies in every experiment, timeseries with labels are produced here.  

**Requirements**:
- raw: .txt-file with information on
    - Start_Cause: Starting time of underlying cause of anomaly (when is the cause first in the system)
    - End_Cause:   Removal time of underlying cause of anomaly  (when is the cause removed from the system)
    - Start_Effect: Starting time of any anomalous effect in the process due to the cause (when does the cause affect the system)
    - End_Effect:   Recovery time of the system (when does the error not affect the system anymore)
    - Anomaly_Class: Is the anomaly a hard fault (1), soft anomaly (2) or controller anomaly (3)?
    - Description:  Details on the anomaly (what happened)
- Processed final sensor data as .csv in 01_Timeseries_Sensor

Please note the following: Given the case one underlying cause is present for a long time and shows itself on several seperate occasions,   
only put the anomaly start_cause / end_cause information in the first occurence of the cause.  
  
For each anomaly, based on the information given on the start and end of cause and effect, three labels are given.  

**Labels**:
- 0: No anomaly, regular operation
- 1: Cause is in the system, no effect
- 2: Cause and effect are both in the system
- 3: Cause is removed, system is recovering, there are still effects

#### Loading of necessary packages

In [ ]:
from pathlib  import Path
from datetime import datetime
import pandas            as pd	
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

#### Initialization of any parameters needed

In [ ]:
# Define the folder of the sensor data for the distillation time-range
folder_sensordata = Path.cwd().parent / "data" / "01_Timeseries_Sensor" / "final" / "Operation"
folder_labels     = Path.cwd().parent / "data" / "00_Timeseries_Label_Anomaly_Metadata" / "raw"
folder_processed  = Path.cwd().parent / "data" / "01_Timeseries_Label_Anomaly_Metadata" / "final"

First, the labels for any experiment present in the "01_Timeseries_Sensor" are processed.

We also provide the possibility to omit certain failure types.
E.g. the leaking of small air bubbles into the destillate pipe is ignored and not seen as an anomaly.

In [ ]:
for file in folder_sensordata.glob("*.csv"):
    # Read out the Experiment name from the filename
    Experiment = file.stem

    # Check if the Experiment exists in the Labels folder
    if not (folder_labels/f"{Experiment}.txt").exists():
        print(f"Experiment {Experiment} has no label file.")
        continue

    # Read the labels
    df_labels         = pd.read_csv(folder_labels/f"{Experiment}.txt", delimiter=";", encoding="utf-8")
    df_labels.columns = df_labels.columns.str.strip()                        # Strip the whitespace from the column names

    # Drop all rows where the Description is "Minor air leakage HV004 and HV005"
    if (df_labels["Description"] == " Minor air leakage HV004 and HV005").any():
        df_labels = df_labels[df_labels["Description"] != " Minor air leakage HV004 and HV005"]
        # Reindex the dataframe
        df_labels.reset_index(drop=True, inplace=True)
        # If the dataframe is empty, create a new dataframe with the same columns but no rows
        if df_labels.empty:
                placeholder_row = (["00:00:00"] * 4 + [0] +  ["-"])
                df_labels       = pd.DataFrame([placeholder_row], columns=df_labels.columns)

    # Drop all rows where the Description is "Negligible controller anomaly"
    if (df_labels["Description"] == " Negligible controller anomaly").any():
        df_labels = df_labels[df_labels["Description"] != " Negligible controller anomaly"]
        # Reindex the dataframe
        df_labels.reset_index(drop=True, inplace=True)
        # If the dataframe is empty, create a new dataframe with the same columns but no rows
        if df_labels.empty:
                placeholder_row = (["00:00:00"] * 4 + [0] +  ["-"])
                df_labels       = pd.DataFrame([placeholder_row], columns=df_labels.columns)

    # Check if the df_labels is empty
    if df_labels.empty:
        print(f"Experiment {Experiment} has no labels.")
        continue

    # Read the sensor data
    df_sensor         = pd.read_csv(file, index_col=None)                          
    df_sensor["Time"] = pd.to_datetime(df_sensor["Time"], format='%H:%M:%S')   # Convert the time columns to datetime objects

    # Find full range of time
    start_time = datetime.combine(datetime.today(), df_sensor["Time"].min().time())
    end_time   = datetime.combine(datetime.today(), df_sensor["Time"].max().time())
    time_range = pd.date_range(start=start_time, end=end_time, freq='s').time

    # Create a dataframe of same length as the time_range with the columns "Time", "Label (common)" and "Label (advanced)" with 0 values
    df_time                                      = pd.DataFrame(index=time_range)
    df_time["Time"]                              = [t.hour * 3600 + t.minute * 60 + t.second for t in time_range]
    df_time["Label (anomaly)"]       = 0
    # df_time["Label (soft fault)"]       = 0
    # df_time["Label (controller fault)"] = 0

    # Strip the whitespace from the time columns and convert them to datetime objects
    for column in df_labels.columns:
        if column == "Description":
            continue

        if df_labels[column].dtype == 'object':
            df_labels[column] = df_labels[column].str.strip()
            df_labels[column] = pd.to_datetime(df_labels[column], format='%H:%M:%S')

    # Convert the label times to scalar values (in seconds of the day)
    start_cause  = df_labels["Start_Cause"].dt.hour  * 3600 + df_labels["Start_Cause"].dt.minute  * 60 + df_labels["Start_Cause"].dt.second
    end_cause    = df_labels["End_Cause"].dt.hour    * 3600 + df_labels["End_Cause"].dt.minute    * 60 + df_labels["End_Cause"].dt.second
    start_effect = df_labels["Start_Effect"].dt.hour * 3600 + df_labels["Start_Effect"].dt.minute * 60 + df_labels["Start_Effect"].dt.second
    end_effect   = df_labels["End_Effect"].dt.hour   * 3600 + df_labels["End_Effect"].dt.minute   * 60 + df_labels["End_Effect"].dt.second

    for i in range(len(df_labels)):
        if df_labels["Anomaly_Class"][i] == 1:
            if start_cause[i] == 0 and end_cause[i] == 0:
                # There is no information on the cause of the fault
                # -> During the whole time range of the effect, the label is 2, we assume the cause is present the whole time
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 2
            elif start_cause[i] == 0 and end_cause[i] != 0:
                # There is no information on the end of the cause of the fault
                df_time.loc[(df_time["Time"] >= start_cause[i]) & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 1
                # -> During the whole time range of the effect, the label is 2, we assume the cause is present the whole time
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 2

            if start_effect[i] == 0 or end_effect[i] == 0:
                # There is no information on the effect of the fault
                # -> During the whole time range of the cause, the label is 1
                df_time.loc[(df_time["Time"] >= start_cause[i])  & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"]  = 1
            if (start_cause[i] != 0 and end_cause[i] != 0) and (start_effect[i] != 0 and end_effect[i]) != 0:   
                # There is information on the cause and effect of the fault
                # When the time is after the start of the cause and before the end of the cause, the label is 1
                # When the time is after the start of the effect and before the end of the cause, the label is 2
                # When the time is after the end of the cause and before the end of the effect, the label is 3
                # When the time is after the start of the effect and before the end of the effect, while the cause is over the label is 3
                df_time.loc[(df_time["Time"] >= start_cause[i])  & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2),  "Label (anomaly)"] = 1
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2),  "Label (anomaly)"] = 2                    
                if start_effect[i] <= end_cause[i]:
                    df_time.loc[(df_time["Time"] >= end_cause[i])    & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 3
                elif start_effect[i] > end_cause[i]:
                    df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 3
        elif df_labels["Anomaly_Class"][i] == 2:
            if start_cause[i] == 0 and end_cause[i] == 0:                
                # There is no information on the cause of the fault
                # -> During the whole time range of the effect, the label is 2, we assume the cause is present the whole time
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 2
            elif start_cause[i] == 0 and end_cause[i] != 0:
                # There is no information on the end of the cause of the fault
                df_time.loc[(df_time["Time"] >= start_cause[i]) & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 1
                # -> During the whole time range of the effect, the label is 2, we assume the cause is present the whole time
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 2
            if start_effect[i] == 0 or end_effect[i] == 0:
                # There is no information on the effect of the fault
                # -> During the whole time range of the cause, the label is 1
                df_time.loc[(df_time["Time"] >= start_cause[i])  & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"]  = 1
            if (start_cause[i] != 0 and end_cause[i] != 0) and (start_effect[i] != 0 and end_effect[i] != 0):  
                # There is information on the cause and effect of the fault
                # When the time is after the start of the cause and before the end of the cause, the label is 1
                # When the time is after the start of the effect and before the end of the cause, the label is 2
                # When the time is after the end of the cause and before the end of the effect, the label is 3
                # When the time is after the start of the effect and before the end of the effect, while the cause is over the label is 3
                df_time.loc[(df_time["Time"] >= start_cause[i])  & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2),  "Label (anomaly)"] = 1
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2),  "Label (anomaly)"] = 2
                if start_effect[i] <= end_cause[i]:
                    df_time.loc[(df_time["Time"] >= end_cause[i])    & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 3
                elif start_effect[i] > end_cause[i]:
                    df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 3
        elif df_labels["Anomaly_Class"][i] == 3:
            if start_cause[i] == 0 and end_cause[i] == 0:
                # There is no information on the cause of the fault
                # -> During the whole time range of the effect, the label is 2, we assume the cause is present the whole time
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 2
                df_time.loc[(df_time["Time"] >= start_cause[i])  & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2),  "Label (anomaly)"] = 1
            elif start_cause[i] == 0 and end_cause[i] != 0:
                # There is no information on the end of the cause of the fault
                df_time.loc[(df_time["Time"] >= start_cause[i]) & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 1
                # -> During the whole time range of the effect, the label is 2, we assume the cause is present the whole time
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 2
            if start_effect[i] == 0 or end_effect[i] == 0:
                # There is no information on the effect of the fault
                # -> During the whole time range of the cause, the label is 1
                df_time.loc[(df_time["Time"] >= start_cause[i])  & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"]  = 1
            if (start_cause[i] != 0 and end_cause[i] != 0) and (start_effect[i] != 0 and end_effect[i] != 0):   
                # There is no information on the effect of the fault
                # -> During the whole time range of the cause, the label is 1
                df_time.loc[(df_time["Time"] >= start_cause[i])  & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2),  "Label (anomaly)"] = 1
                df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_cause[i]) & (df_time["Label (anomaly)"] != 2),  "Label (anomaly)"] = 2
                if start_effect[i] <= end_cause[i]:
                    df_time.loc[(df_time["Time"] >= end_cause[i])    & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 3
                elif start_effect[i] > end_cause[i]:
                    df_time.loc[(df_time["Time"] >= start_effect[i]) & (df_time["Time"] <= end_effect[i]) & (df_time["Label (anomaly)"] != 2), "Label (anomaly)"] = 3

    # Switch index to column "Time"
    df_time.reset_index(inplace=True)
    df_time.drop(columns=["Time"], inplace=True)
    df_time.rename(columns={"index": "Time"}, inplace=True)

    # Save the dataframe to a csv file without the index
    df_time.to_csv(folder_processed/f"{Experiment}.csv", index=False, encoding="utf-8")
    print(f"Experiment {Experiment} has been processed.")